# Notebook : De la Chaîne de Caractères à l'Espace Vectoriel

L'objectif de ce notebook est de comprendre comment nous transformons un texte continu $S$ en une suite discrète d'indices $T \in \{1, \dots, |V|\}^N$, où $|V|$ est la taille du vocabulaire.Nous cherchons un optimum entre deux extrêmes :
- Tokenisation par caractère : $|V|$ est petit ($\approx 100$), mais $N$ est très grand. Le modèle doit apprendre à épeler avant d'apprendre le sens.
- Tokenisation par mot : $N$ est petit, mais $|V|$ tend vers l'infini (problème des mots inconnus ou "OOV").

L'état de l'art (Subword Tokenization) se situe entre les deux.

In [ ]:
import torch
import torch.nn as nn
from collections import defaultdict
import re

# Corpus jouet pour l'exercice
corpus = [
    "le mathématicien calcule",
    "le calcul est bon",
    "le mathématicien est content",
    "calculer est essentiel"
]

## Partie 1 : L'approche Naïve et ses limites
Commençons par une tokenisation par mot simple.L'opération mathématique est une bijection $f: \text{Vocabulaire} \to \mathbb{N}$.

In [ ]:
# Construction du vocabulaire naïf
word_freqs = defaultdict(int)
for sentence in corpus:
    for word in sentence.split():
        word_freqs[word] += 1

vocab = sorted(word_freqs.keys())
word2id = {w: i for i, w in enumerate(vocab)}

print(f"Taille du vocabulaire : {len(vocab)}")
print(f"Mapping : {word2id}")

# Test de robustesse
test_sentence = "le mathématicien calcule vite"
try:
    tokens = [word2id[w] for w in test_sentence.split()]
    print(tokens)
except KeyError as e:
    print(f"Erreur : Le token {e} est hors vocabulaire (OOV).")

Analyse : Le mot "vite" casse le modèle. C'est inacceptable pour un LLM généraliste.

## Partie 2 : Implémentation du BPE (Byte Pair Encoding)
C'est ici que tu travailles. Le BPE est un algorithme itératif de compression. L'intuition : Remplacer les paires de caractères (ou de bytes) les plus fréquentes par un nouveau symbole, jusqu'à atteindre une taille de vocabulaire cible.

Nous allons partir d'un vocabulaire constitué uniquement de caractères, puis fusionner.

Étape 2.1 : Pré-tokenisation
On représente chaque mot comme une liste de caractères, avec un symbole spécial /w pour marquer la fin du mot (essentiel pour distinguer "est" de "est" dans "test").

In [ ]:
# Initialisation : dictionnaire de fréquences des mots pré-tokenisés
# Format : "c a l c u l </w>" : fréquence
vocab_bpe = defaultdict(int)
for sentence in corpus:
    for word in sentence.split():
        # Espace entre chaque char pour le traitement
        word_spaced = " ".join(list(word)) + " </w>"
        vocab_bpe[word_spaced] += 1

print("État initial :", dict(vocab_bpe))

Étape 2.2 : Le Cœur du Réacteur (À toi de coder)

Tu dois implémenter la fonction qui trouve la paire de tokens adjacents la plus fréquente dans tout le corpus.

In [ ]:
def get_stats(vocab):
    """
    Calcule la fréquence de chaque paire de symboles adjacents.
    Input: dictionnaire {mot_espacé: fréquence}
    Output: dictionnaire {(symbole1, symbole2): fréquence_totale}
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        # --- DÉBUT CODE ÉTUDIANT ---
        # Itérer sur les symboles pour compter les bigrammes adjacents
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] += freq
        # --- FIN CODE ÉTUDIANT ---
    return pairs

def merge_vocab(pair, v_in):
    """
    Fusionne la paire la plus fréquente dans le vocabulaire.
    Si pair = ('e', 's'), remplace toutes les occurrences de 'e s' par 'es'.
    """
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    
    for word in v_in:
        # Remplace le bigramme par le token fusionné
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Testons ton implémentation sur une itération
pairs = get_stats(vocab_bpe)
if not pairs:
    print("Erreur: Pas de paires trouvées. Vérifie ton code.")
else:
    best = max(pairs, key=pairs.get)
    print(f"Paire la plus fréquente : {best} avec {pairs[best]} occurrences")
    
    # Mise à jour du vocabulaire
    vocab_bpe = merge_vocab(best, vocab_bpe)
    print("Vocabulaire après 1 fusion :", dict(vocab_bpe))

Étape 2.3 : Entraînement complet

Lançons l'algorithme pour $K$ fusions. Observe comment les "caractères" deviennent des "sous-mots" puis des "racines".

In [ ]:
num_merges = 10
merge_history = []

for i in range(num_merges):
    pairs = get_stats(vocab_bpe)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab_bpe = merge_vocab(best, vocab_bpe)
    merge_history.append(best)
    print(f"Fusion {i+1}: {best}")

print("\nTokens finaux appris (exemples) :")
print(list(vocab_bpe.keys()))

## Partie 3 : Utilisation Industrielle (Hugging Face)
Maintenant que tu as compris la mécanique interne, utilisons une implémentation optimisée en Rust (tokenizers via transformers). Nous allons comparer la tokenisation d'un modèle BERT (WordPiece) et d'un GPT (Byte-level BPE).

In [ ]:
from transformers import AutoTokenizer

# Chargement d'un tokenizer pré-entraîné (CamemBERT pour le français)
tokenizer = AutoTokenizer.from_pretrained("camembert-base")

text = "L'anticonstitutionnellement est un mot complexe."

# Encodage
encoded = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'])

print(f"Texte original : {text}")
print(f"Tokens : {tokens}")
print(f"IDs : {encoded['input_ids']}")

## Partie 4 : De l'Index au Vecteur (Embeddings)
La tokenisation nous donne des indices $i \in \mathbb{N}$. Le modèle a besoin de vecteurs $x \in \mathbb{R}^d$.C'est le rôle de la couche d'Embedding (une simple Lookup Table différentiable).$$E: \{1, \dots, |V|\} \to \mathbb{R}^d$$

In [ ]:
vocab_size = tokenizer.vocab_size
embedding_dim = 64 # Dimension arbitraire pour l'exemple

# Création de la couche
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Transformation des IDs en tenseurs
input_tensor = torch.tensor(encoded['input_ids']).unsqueeze(0) # Batch size 1
vectorized_sequence = embedding_layer(input_tensor)

print(f"Shape des IDs : {input_tensor.shape}") # (1, Seq_Len)
print(f"Shape des vecteurs : {vectorized_sequence.shape}") # (1, Seq_Len, 64)

# Vérification du gradient
print(f"Requires Grad : {vectorized_sequence.requires_grad}")

## Défi Final

Le BPE est glouton (greedy). Il prend la meilleure paire localement.Il existe une méthode probabiliste appelée Unigram Language Model (utilisée par SentencePiece/T5) qui initialise un gros vocabulaire et supprime les tokens qui minimisent le moins la perte de vraisemblance du corpus.

Ta mission : Sans le coder, écris en LaTeX la fonction de perte (Likelihood) qu'on chercherait à maximiser si on considère que la phrase $S$ est une séquence de tokens indépendants $t_1, \dots, t_M$.

(Indice : C'est le produit des probabilités d'apparition de chaque sous-mot).

## Impact de la Tokenisation sur la Dynamique d'Apprentissage
Nous allons démontrer empiriquement une loi fondamentale des LLM : augmenter la taille du vocabulaire permet de réduire la longueur de séquence $N$, ce qui est critique car l'attention est en complexité quadratique $O(N^2)$.
### Partie 5 : Le Laboratoire (Setup)
Nous définissons un modèle Transformer "agnostique" qui accepte n'importe quelle taille de vocabulaire.Hypothèse mathématique à vérifier :Soit $k$ le taux de compression moyen du BPE par rapport aux caractères ($N_{bpe} \approx N_{char} / k$).
- Le coût d'inférence par étape temporelle (Forward pass) pour l'attention chute de $N^2$ à $(N/k)^2$.
- L'entropie par token (densité d'information) augmente, rendant la prédiction unitaire plus "dure" (perplexité plus élevée par token), mais la convergence sur la séquence globale plus rapide.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import math

# --- Modèle Minimaliste (Mini-GPT) ---
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, context_length):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_length, embed_dim)
        
        # Couche Transformer standard
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=embed_dim*4, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.lm_head = nn.Linear(embed_dim, vocab_size)
        self.context_length = context_length

    def forward(self, idx):
        B, T = idx.shape
        
        # 1. Embeddings + Positional
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        
        # 2. Masque Causal (Empêcher de voir le futur)
        # mask[i, j] = -inf si j > i
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T).to(idx.device)
        
        # 3. Transformer
        x = self.transformer(x, mask=causal_mask, is_causal=True)
        
        # 4. Projection finale
        logits = self.lm_head(x)
        return logits

# Hyperparamètres fixes pour la comparaison
EMBED_DIM = 64
NUM_HEADS = 4
LAYERS = 2
CONTEXT_LEN = 128 # Fenêtre d'attention
BATCH_SIZE = 32
STEPS = 500
LEARNING_RATE = 3e-3

### Partie 6 : Le Match (Char vs BPE)
Nous allons simuler une tâche de modélisation de langage sur un corpus de texte synthétique (ou un extrait de livre).

Préparation des Tokenizers :

In [ ]:
# 1. Données : Un texte assez long (ex: Les Misérables ou code source)
# Pour l'exemple, on répète un corpus synthétique
raw_text = (
    "L'algèbre linéaire est fondamentale pour l'apprentissage profond. "
    "La décomposition en valeurs singulières permet de factoriser des matrices. "
    "Les réseaux de neurones sont des approximateurs universels. "
    "La tokenisation réduit la dimensionnalité séquentielle. "
) * 500

# 2. Tokenizer A : Caractère
chars = sorted(list(set(raw_text)))
stoi_char = {ch:i for i,ch in enumerate(chars)}
itos_char = {i:ch for i,ch in enumerate(chars)}
vocab_size_char = len(chars)

def encode_char(s): return [stoi_char[c] for c in s]
data_char = torch.tensor(encode_char(raw_text), dtype=torch.long)

# 3. Tokenizer B : BPE (Utilisons celui de HF pour la robustesse)
from transformers import AutoTokenizer
# On triche un peu en prenant un tokenizer déjà entrainé pour gagner du temps, 
# mais le principe est le même : vocabulaire ~30k
tokenizer_bpe = AutoTokenizer.from_pretrained("gpt2") 
data_bpe = torch.tensor(tokenizer_bpe.encode(raw_text), dtype=torch.long)
vocab_size_bpe = tokenizer_bpe.vocab_size

print(f"Comparaison des données :")
print(f"Longueur Char Dataset: {len(data_char)}")
print(f"Longueur BPE Dataset : {len(data_bpe)}")
print(f"Facteur de compression k : {len(data_char) / len(data_bpe):.2f}")

In [ ]:
def train_model(model_name, vocab_size, data, device='cpu'):
    print(f"\n--- Entraînement {model_name} (Vocab: {vocab_size}) ---")
    model = MiniTransformer(vocab_size, EMBED_DIM, NUM_HEADS, LAYERS, CONTEXT_LEN).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    start_time = time.time()
    losses = []
    
    model.train()
    for step in range(STEPS):
        # Sampling aléatoire de batchs
        ix = torch.randint(len(data) - CONTEXT_LEN, (BATCH_SIZE,))
        x = torch.stack([data[i:i+CONTEXT_LEN] for i in ix]).to(device)
        y = torch.stack([data[i+1:i+CONTEXT_LEN+1] for i in ix]).to(device)
        
        logits = model(x)
        
        # Reshape pour CrossEntropy : (B*T, Vocab)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if step % 100 == 0:
            print(f"Step {step}: Loss {loss.item():.4f}")
            
    dt = time.time() - start_time
    print(f"Entraînement terminé en {dt:.2f}s")
    return losses, model

# Lancer le match (si GPU dispo, change device='cuda')
losses_char, model_char = train_model("Char-Level", vocab_size_char, data_char)
losses_bpe, model_bpe = train_model("BPE-Level", vocab_size_bpe, data_bpe)

### Partie 7 : Analyse des Résultats (Le "So What?")
Ici, nous devons être rigoureux. On ne peut pas comparer directement la Loss (Cross-Entropy) des deux modèles car elles ne vivent pas dans le même espace probabiliste.
- Loss Char : $-\log P(c_t | c_{<t})$. Espace de sortie $\approx 80$ classes.
- Loss BPE : $-\log P(tok_t | tok_{<t})$. Espace de sortie $\approx 50 000$ classes.

La loss du BPE sera naturellement plus élevée au départ (plus dur de deviner 1 mot parmi 50k que 1 lettre parmi 80). Pourtant, le modèle BPE apprend "plus vite" sémantiquement.

Visualisation Normalisée (Bits per Character) :Pour comparer, on doit ramener la mesure à une unité commune : le bit par caractère.$$\text{BPC} = \frac{\text{Loss}_{\text{token}}}{\ln(2)} \times \frac{1}{\text{Compression Ratio } (k)}$$

Note : Pour le modèle Char, $k=1$.Python

In [ ]:
import matplotlib.pyplot as plt

# Calcul du ratio de compression effectif
k_compression = len(data_char) / len(data_bpe)

# Conversion en Bits par Caractère (log base 2)
# Loss (nats) / ln(2) = Loss (bits)
bpc_char = [l / math.log(2) for l in losses_char]
bpc_bpe = [(l / math.log(2)) / k_compression for l in losses_bpe]

plt.figure(figsize=(10, 5))
plt.plot(bpc_char, label='Char-Level Model', alpha=0.7)
plt.plot(bpc_bpe, label='BPE Model (Normalized)', linewidth=2)
plt.xlabel("Steps d'entraînement")
plt.ylabel("Bits per Character (Plus bas est mieux)")
plt.title(f"Convergence Normalisée (Ratio compression k={k_compression:.2f})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Partie 8 : L'inconvénient caché du BPE (Tokenization Artefacts)
C'est bien beau l'efficacité, mais le BPE introduit des bugs cognitifs dans les LLM. Comme le modèle voit des "chunks", il devient aveugle à la composition interne des mots.

Démonstration : Demande au modèle d'inverser une chaîne de caractères ou de compter les lettres.

Si le token est [Calcul] (ID 4521), le modèle n'a aucun moyen arithmétique de savoir qu'il contient 6 lettres ou qu'il finit par 'l', sauf s'il a mémorisé cette corrélation statistiquement.

Le modèle Char-level, lui, voit [C, a, l, c, u, l]. Il trivialise les tâches morphologiques.

C'est pour cette raison que GPT-4 est mauvais pour faire des anagrammes ou compter des caractères ("Combien de 'r' dans strawberry ?").

Exercice Mental pour l'X : Si tu devais concevoir un modèle pour des mathématiques pures (ex: démonstration de théorèmes formels), quel tokenizer choisirais-tu ?

Indice : En maths, x et y sont des entités atomiques, mais sin est une fonction. 123 est un entier, mais on veut parfois traiter les digits.

Réponse attendue : Souvent un hybride ou un tokenizer caractère spécialisé. Pour du code (Python), on garde les mots-clés (def, return) mais on fragmente fortement les identifiants inconnus.